In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 12
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
15.576614640640418

Trial 1 =========================================
15.324995273737006

Trial 2 =========================================
14.619789702555286

Trial 3 =========================================
15.944409049678583

Trial 4 =========================================
15.339111316338773

Trial 5 =========================================
14.614125916509561

Trial 6 =========================================
15.388950245520386

Trial 7 =========================================
18.790262140292164

Trial 8 =========================================
14.294357143062124



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 9 =========================================
16.08748501292667

Trial 10 =========================================
17.03236382537954



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 11 =========================================
16.216815755134846

Trial 12 =========================================
15.306869447991122

Trial 13 =========================================
16.383934911779782

Trial 14 =========================================
14.43684894845585

Trial 15 =========================================
14.129450024819064

Trial 16 =========================================
14.151488255333877

Trial 17 =========================================
15.831469072335226

Trial 18 =========================================
16.096935490825807

Trial 19 =========================================
15.856825059056641

Trial 20 =========================================
15.181272170194827

Trial 21 =========================================
17.417442597094123

Trial 22 =========================================
15.359973061557032

Trial 23 =========================================
16.232075291430068

Trial 24 =========================================
15.280346595116203

Trial 2

...

Trial 88 =========================================
15.975726326149227

Trial 89 =========================================
15.380726716636362

Trial 90 =========================================
14.287238787138238



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/mi

Trial 91 =========================================
16.256010606759

Trial 92 =========================================
16.398950825132818

Trial 93 =========================================
13.80274797628588

Trial 94 =========================================
16.232693772443177

Trial 95 =========================================
15.366531952630533

Trial 96 =========================================
15.39690840762643

Trial 97 =========================================
14.2517777209931



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 98 =========================================
16.063216425164416

Trial 99 =========================================
15.382253360524949

[15.57661464 15.32499527 14.6197897  15.94440905 15.33911132 14.61412592
 15.38895025 18.79026214 14.29435714 16.08748501 17.03236383 16.21681576
 15.30686945 16.38393491 14.43684895 14.12945002 14.15148826 15.83146907
 16.09693549 15.85682506 15.18127217 17.4174426  15.35997306 16.23207529
 15.2803466  15.35977252 15.9903061  18.81686241 16.17850404 14.50255208
 16.16132247 14.47110943 16.05140939 14.41196709 14.41758413 15.32906972
 15.37125509 16.2302237  17.57056649 16.24263325 13.36927958 14.42903596
 14.25482559 14.22987354 17.26599376 14.37217577 15.59509586 14.50791922
 15.36376099 15.92922201 16.05455226 18.77601016 14.85562457 14.89074059
 17.56765563 14.13076774 15.95816252 14.28606514 14.68623156 13.88264701
 15.37596747 14.61382404 13.32029503 16.18022373 15.3905425  16.21120698
 16.50356951 15.39680233 16.22218732 18.65740877 13.928

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.81686241258938
Avg = 15.553590950122123
Std = 1.2123874572868552


In [6]:
print(y_max_arr.tolist())

[15.576614640640418, 15.324995273737006, 14.619789702555286, 15.944409049678583, 15.339111316338773, 14.614125916509561, 15.388950245520386, 18.790262140292164, 14.294357143062124, 16.08748501292667, 17.03236382537954, 16.216815755134846, 15.306869447991122, 16.383934911779782, 14.43684894845585, 14.129450024819064, 14.151488255333877, 15.831469072335226, 16.096935490825807, 15.856825059056641, 15.181272170194827, 17.417442597094123, 15.359973061557032, 16.232075291430068, 15.280346595116203, 15.359772524124006, 15.990306099585712, 18.81686241258938, 16.17850403525088, 14.50255207703659, 16.16132246543883, 14.471109431760055, 16.051409385298655, 14.411967094956543, 14.41758412794891, 15.32906972437609, 15.371255086766409, 16.2302236987753, 17.570566491856145, 16.242633252117766, 13.369279582773599, 14.429035959468067, 14.254825592011045, 14.2298735355367, 17.265993762310586, 14.37217576757845, 15.595095862885737, 14.507919221109319, 15.36376098980332, 15.929222010399386, 16.05455226024

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-12.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)